# nanoGPT: pre-train on CodeParrot, fine-tune a classifier on xCodeEval (compiles / compile error)

Edit **only the config cell** between runs.

**Pipeline**
1. Pre-train a small GPT (next-token prediction) on CodeParrot Python files.
2. Build a labelled set from xCodeEval: Python submissions that compile (label 1) or give a compilation error (label 0).
3. Put a classification head on top of the GPT and train it on the labelled set (pool the token states -> head -> 2 classes; pooling = mean / last token / learnable attention).
4. Compare how well this works for different block sizes, pooling methods, head depths, numbers of frozen layers and amounts of labelled data. Every epoch's metrics are written to CSV.

## What is being compared

Every fine-tuning run is one point in this grid:

| Axis | Values | Meaning |
|---|---|---|
| **Block size** (`BLOCK_SIZES`) | 256, 512, 1024 | Context length of the GPT *and* the maximum program length (in tokens) kept in the labelled set. Each block size has its own pre-training runs and its own label file `labels_xcodeeval_b{block}.json` |
| **Positional encoding** (`POS_ENCS`) | rope, abs, nope | One pre-training run per (block size, encoding) |
| **Pooling** (`POOLINGS`) | mean, last, attn | How the token states become one vector: mean over real tokens, the last real token, or learnable attention pooling (one query vector) |
| **Initialisation** | pre-trained / scratch | Start from the CodeParrot-pre-trained GPT, or from random weights (the "does pre-training help at all?" baseline) |
| **Frozen blocks** (`n_freeze`) | `0`, `N_LAYER//2`, `N_LAYER` | How many of the *bottom* transformer blocks keep their pre-trained weights fixed. `0` = full fine-tune. `N_LAYER` = fully frozen GPT, only the head (and the attention-pool query) learns. Whenever `n_freeze > 0` the token (and position) embeddings are frozen as well |
| **Head depth** | 1, 3, 5, 10 | 1 = a single linear layer. Deeper heads repeat `Linear -> GELU -> Dropout` (depth-1) times before the final linear layer (hidden width `HEAD_HIDDEN`) |
| **Labelled examples** | 10, 100, 1000, 10,000, "full" | Class-balanced subsets of the labelled training pool; `"full"` = the whole pool (not balanced). Numbers larger than the largest balanced set are clamped to it |
| **Seed** | `GRID_SEEDS` = 0, 1, 2 | Repeats to separate real differences from noise |

The scratch baseline uses the same pooling, head depths and label sizes but is always fully trainable (`n_freeze = 0`).

Pre-training length is rescaled so every block size sees about the same number of tokens (256 -> 10,000 iterations, 512 -> 5,000, 1024 -> 2,500).

Results: one CSV per (positional encoding, block size), `grid_results_<pe>_b<block>.csv`, with **one row per epoch** of every run.


In [ ]:
# ======================================================================
# CONFIG - the only cell you should need to edit between runs
# ======================================================================
import os, torch

PRESET   = "gpu"
POS_ENCS = ("rope", "abs", "nope")   # nope = control: does position info matter at all?

# PRESET  = "gpu"     # "cpu" = the small laptop run you did before | "gpu" = bigger run for a cloud GPU
# POS_ENCS = ("rope", "abs")   # positional encodings to compare; each is pre-trained and evaluated in turn.
                             # options: "rope" = rotary, "abs" = GPT-2 learned absolute, "nope" = none
POS_ENC  = POS_ENCS[0]       # used by the single-run section; the pre-training and grid loops switch it automatically
BLOCK_SIZES = (256, 512, 1024)            # context length = max program length in tokens; each gets its own pre-training runs + label file
POOLINGS    = ("mean", "last", "attn")    # mean = mean of real tokens | last = last real token | attn = learnable attention pooling
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True   # faster fp32 matmuls on Ampere+ GPUs (fine-tuning runs in fp32)
    torch.backends.cudnn.allow_tf32 = True

# ---- Size of the experiment (chosen by PRESET) ----
# BATCH_SIZE / GRAD_ACCUM / PRETRAIN_ITERS below are the values for 256-token blocks; iters_for/batch_for/accum_for
# (defined after this block) derive the values for the other block sizes.
# BLOCK_SIZE (no S) is the *current* block size; use_config() in section 5 switches it. It is also the max length
# of an xCodeEval program (longer ones are dropped when building labels): labels_xcodeeval_b{block}.json per block size.
if PRESET == "cpu":
    PRETRAIN_FILES, PRETRAIN_ITERS = 5000, 3000        # ~9M tokens seen, <1 pass over the data
    BATCH_SIZE, GRAD_ACCUM = 12, 1
    N_LAYER, N_HEAD, N_EMBD = 4, 4, 128                # ~7M parameters
    XC_MAX_SHARDS = 8
    GRID_SEEDS = (0,)
elif PRESET == "gpu":
    PRETRAIN_FILES, PRETRAIN_ITERS = 80_000, 10_000    # ~350M tokens, ~1 pass -> less overfitting than 3.7 passes
    BATCH_SIZE, GRAD_ACCUM = 64, 2    # micro-batch / accumulation at 256-token blocks
    N_LAYER, N_HEAD, N_EMBD = 8, 8, 512
    XC_MAX_SHARDS = 400
    GRID_SEEDS = (0, 1, 2)

    # PRETRAIN_FILES, PRETRAIN_ITERS = 20_000, 10_000    # ~180M tokens of data, ~330M tokens seen (~2 passes)
    # BATCH_SIZE, GRAD_ACCUM = 64, 2    # 16,384 tokens per iteration
    # N_LAYER, N_HEAD, N_EMBD = 8, 8, 512                # ~50M parameters (about half is the GPT-2 embedding table)
    # XC_MAX_SHARDS = 400                                 # more shards -> enough compile-error examples (the rare class)
    # GRID_SEEDS = (0, 1, 2)                             # drop to (0,) for a quick first pass

    # PRETRAIN_FILES, PRETRAIN_ITERS = 500, 300          # SMOKE TEST (full: 40_000, 20_000)
    # BATCH_SIZE, GRAD_ACCUM = 32, 2    # 16,384 tokens per iteration
    # N_LAYER, N_HEAD, N_EMBD = 8, 8, 512                # ~50M parameters
    # XC_MAX_SHARDS = 40                                 # SMOKE TEST (full: 40)
    # GRID_SEEDS = (0,)                                  # SMOKE TEST (full: (0, 1, 2), or (0,) for a first pass)
else:
    raise ValueError("PRESET must be 'cpu' or 'gpu'")

# ---- Pre-training schedule per block size: keep the total tokens seen roughly constant ----
# gpu preset: 256 -> 10,000 iters | 512 -> 5,000 | 1024 -> 2,500. Sequences per iteration stay BATCH_SIZE*GRAD_ACCUM, so
# tokens per iteration grow with the block size and the total (iters x tokens/iter) stays ~327M.
# SCALE_MICROBATCH: shrink the micro-batch / raise grad-accum for longer blocks so GPU memory per forward pass stays
# the same as at 256 tokens (64x256 -> 32x512 -> 16x1024 tokens-per-micro-batch = 16,384). Set False if you have the memory.
SCALE_MICROBATCH = True
def iters_for(blk): return PRETRAIN_ITERS * 256 // blk
def batch_for(blk): return max(1, BATCH_SIZE * 256 // blk) if SCALE_MICROBATCH else BATCH_SIZE
def accum_for(blk): return GRAD_ACCUM * blk // 256 if SCALE_MICROBATCH else GRAD_ACCUM
BLOCK_SIZE = BLOCK_SIZES[0]        # "current" block size, switched by use_config()

# ---- xCodeEval labelled data (built once, see "Build labels") ----
XC_LANG_CLUSTER = "Python"
XC_N_PER_CLASS  = 5000      # stop collecting once we have this many examples per class
def label_file(blk): return f"labels_xcodeeval_b{blk}.json"   # one label file per block size

# ---- Fine-tuning (classifier), used by the "single runs" section ----
HEAD_DEPTH  = 1             # number of linear layers in the head: any depth >= 1
HEAD_HIDDEN = 256           # hidden width of the head (ignored when HEAD_DEPTH = 1)
N_LABELED   = 4800          # labelled training examples used (None = whole pool), class-balanced

POOLING     = "mean"        # pooling used by the single-run section: "mean", "last" or "attn"
FT_EPOCHS   = 3
#FT_EPOCHS   = 10

N_FREEZE    = 0             # bottom blocks to freeze: 0 = full fine-tune, N_LAYER = whole GPT frozen
LR_HEAD     = 1e-3          # learning rate of the (randomly initialised) head
LR_BACKBONE = 5e-5          # CHANGED (was 1e-4): gentler -> less overfitting of the pretrained backbone
LR_BACKBONE_SCRATCH = 3e-4  # CHANGED (was 1e-4): random init needs a higher LR -> fewer collapsed scratch runs

# ---- Fine-tuning (classifier), used by the "single runs" section ----
# HEAD_DEPTH  = 1             # number of linear layers in the head: any depth >= 1
# HEAD_HIDDEN = 256           # hidden width of the head (ignored when HEAD_DEPTH = 1)
# N_LABELED   = 4800          # labelled training examples used (None = whole pool), class-balanced
# FT_EPOCHS   = 10
# N_FREEZE    = 0             # bottom blocks to freeze: 0 = full fine-tune, N_LAYER = whole GPT frozen
# LR_HEAD     = 1e-3          # learning rate of the (randomly initialised) head
# LR_BACKBONE = 1e-4          # learning rate of the pre-trained GPT blocks that are not frozen
# LR_BACKBONE_SCRATCH = 1e-4  # same, but for the random-init baseline (it may want a higher LR than 1e-4)

# ---- Experiment grid (section 7) ----

GRID_FREEZE = (0, N_LAYER // 2, N_LAYER)          # full fine-tune / half frozen (best so far) / linear probe
GRID_DEPTHS = (1, 3, 5, 10)                       # number of linear layers in the head
GRID_SIZES  = (10, 100, 1000, 10_000, "full")     # "full" = whole train pool (not balanced); ints are balanced and clamped to the largest balanced set
GRID_EPOCHS = 3       

# GRID_FREEZE = sorted({0, N_LAYER // 4, N_LAYER // 2, N_LAYER})   # e.g. 8 layers -> 0, 2, 4, 8
# GRID_DEPTHS = (1, 2, 3)
# GRID_SIZES  = (500, 1000, 2500, 5000)
# GRID_EPOCHS = 5

# GRID_FREEZE = sorted({0, N_LAYER // 4, N_LAYER // 2, N_LAYER})   # 8 layers -> 0, 2, 4, 8
# GRID_DEPTHS = (1,)                 # SMOKE TEST (full: (1, 2, 3))
# GRID_SIZES  = (500,)               # SMOKE TEST (full: (500, 1000, 2500, 5000))
# GRID_EPOCHS = 5
def results_csv(pe, blk): return f"grid_results_{pe}_b{blk}.csv"   # per-epoch rows are appended here as runs finish

REUSE_PRETRAINED = False   # True: skip pre-training for a (block size, encoding) whose checkpoint already exists (e.g. after an interrupted run)
def pretrain_dir(pe, blk):  # where train.py writes ckpt.pt for positional encoding `pe` and block size `blk`
    return f"out-code-{pe}-b{blk}-{iters_for(blk)}it-{PRETRAIN_FILES}f"
print(f"device={DEVICE} preset={PRESET} positional encodings={POS_ENCS} block sizes={BLOCK_SIZES} pooling={POOLINGS}")
for blk in BLOCK_SIZES:
    tpi = batch_for(blk) * blk * accum_for(blk)
    print(f"block {blk:4d}: {iters_for(blk):6,} iters x {tpi:,} tokens/iter = {tpi*iters_for(blk)/1e6:.1f}M tokens seen "
          f"(micro-batch {batch_for(blk)} x accum {accum_for(blk)})")


## 1. Build labels from xCodeEval (run once; label 1 = compiles, 0 = compilation error)

Runs first so you know early how many compile-error examples there are. Compile errors are rare (roughly 20-50 per shard), so this needs many shards.

In [ ]:
import os, json, random, time, tiktoken
from huggingface_hub import list_repo_files, hf_hub_download

enc = tiktoken.get_encoding("gpt2")      # same tokenizer as the rest of the notebook (needed here for the length filter)

# One pass over the shards builds the label file of every block size that is still missing
# (each program is tokenised once; it goes into every block size whose context window it fits in).
missing = [blk for blk in BLOCK_SIZES if not os.path.exists(label_file(blk))]
for blk in BLOCK_SIZES:
    if blk not in missing:
        print(f"{label_file(blk)} already exists - reusing it (delete the file to rebuild)")

if missing:
    REPO = "NTU-NLP-sg/xCodeEval"
    random.seed(0)
    t0 = time.time()

    # List the code_compilation training shards (jsonl files) and visit them in random order.
    shards = sorted(f for f in list_repo_files(REPO, repo_type="dataset")
                    if f.startswith("code_compilation/train/") and f.endswith(".jsonl"))
    print(len(shards), "train shards, e.g.", shards[:2])
    random.shuffle(shards)

    pos = {blk: [] for blk in missing}     # pos = compiles (label 1), neg = compilation error (label 0)
    neg = {blk: [] for blk in missing}
    for shard in shards[:XC_MAX_SHARDS]:
        path = hf_hub_download(REPO, shard, repo_type="dataset")
        with open(path, encoding="utf-8") as f:
            for line in f:
                ex = json.loads(line)
                # keep Python 3 / PyPy 3 only (Python 2 code would "fail to compile" for the wrong reason)
                if ex["lang_cluster"] != XC_LANG_CLUSTER or "3" not in ex["lang"]:
                    continue
                code = ex["source_code"]
                n_tok = len(enc.encode_ordinary(code))
                label = 0 if ex["compilation_error"] else 1
                for blk in missing:
                    if n_tok > blk:                                    # whole program must fit in the context window
                        continue
                    bucket = pos[blk] if label == 1 else neg[blk]
                    if len(bucket) < XC_N_PER_CLASS:
                        # "group" = problem id; used later to keep the same problem out of both train and test
                        bucket.append({"code": code, "label": label, "group": ex["src_uid"]})
        print(f"{shard}: " + " | ".join(f"b{blk}: pos={len(pos[blk])} neg={len(neg[blk])}" for blk in missing)
              + f"  ({(time.time()-t0)/60:.1f} min)")
        if all(len(pos[blk]) >= XC_N_PER_CLASS and len(neg[blk]) >= XC_N_PER_CLASS for blk in missing):
            break

    for blk in missing:
        json.dump(pos[blk] + neg[blk], open(label_file(blk), "w", encoding="utf-8"))
        print(f"saved {label_file(blk)}:", len(pos[blk]) + len(neg[blk]), "examples;", len(pos[blk]), "compile /", len(neg[blk]), "error")
    # Compile errors are the rare class. If neg is much smaller than XC_N_PER_CLASS, raise XC_MAX_SHARDS:
    # class balancing later is limited by the smaller class.


## 2. Pre-training data (CodeParrot)

In [ ]:
import os, numpy as np, tiktoken
from datasets import load_dataset

enc = tiktoken.get_encoding("gpt2")     # GPT-2 BPE tokenizer (vocab 50,257); also used later for the labelled code
out = "data/codeparrot"                 # relative to the nanoGPT folder; train.py reads data/<dataset>/{train,val}.bin
os.makedirs(out, exist_ok=True)

def dump(name, split, max_files):
    # Stream `max_files` files from the HF dataset, tokenize each, and join them into one flat uint16 array,
    # with an end-of-text token between files. (nanoGPT's .bin format.)
    ds = load_dataset(name, split=split, streaming=True)
    chunks = []                          # one small numpy array per file (a python list of ints would use ~8 GB at this scale)
    for i, ex in enumerate(ds):
        if i >= max_files: break
        chunks.append(np.array(enc.encode_ordinary(ex["content"]) + [enc.eot_token], dtype=np.uint16))
    return np.concatenate(chunks)

train = dump("codeparrot/codeparrot-clean", "train", PRETRAIN_FILES)
val   = dump("codeparrot/codeparrot-clean-valid", "train", 500)   # held-out files for the pre-training val loss
train.tofile(os.path.join(out, "train.bin"))
val.tofile(os.path.join(out, "val.bin"))
print(f"{len(train):,} train tokens, {len(val):,} val tokens")
tokens_seen = max(batch_for(blk) * blk * accum_for(blk) * iters_for(blk) for blk in BLOCK_SIZES)
print(f"pre-training will make {tokens_seen/len(train):.2f} passes over the data (same for every block size)")

## 3. Pre-train

In [ ]:
# Pre-trains one GPT per (block size, positional encoding), one after the other (language-model pre-training from scratch).
# Each checkpoint goes to pretrain_dir(pe, blk)/ckpt.pt, which the classifier loads in section 4/5.
# --compile=False avoids torch.compile problems with the custom RoPE/position code; you can try =True on the GPU for speed.
# --warmup_iters: LR warm-up over the first 5% of iterations, then cosine decay to the end (lr_decay_iters = max_iters).
# Iterations / micro-batch / grad-accum come from iters_for / batch_for / accum_for so every block size sees ~the same number of tokens.
# Same data, model size, seed and schedule shape for every encoding, so only the positional encoding differs within a block size.
import gc
for blk in BLOCK_SIZES:
    for pe in POS_ENCS:
        pre_dir, iters = pretrain_dir(pe, blk), iters_for(blk)
        if REUSE_PRETRAINED and os.path.exists(f"{pre_dir}/ckpt.pt"):
            print(f"[{pe} b{blk}] checkpoint exists, skipping pre-training"); continue
        print(f"\n===== pre-training: block_size={blk} pos_enc={pe} iters={iters} -> {pre_dir} =====")
        get_ipython().run_line_magic("run",
            f"train.py --dataset=codeparrot --out_dir={pre_dir} "
            f"--device={DEVICE} --compile=False --pos_enc={pe} --warmup_iters={iters // 20} "
            f"--block_size={blk} --batch_size={batch_for(blk)} --gradient_accumulation_steps={accum_for(blk)} "
            f"--n_layer={N_LAYER} --n_head={N_HEAD} --n_embd={N_EMBD} --dropout=0.0 "
            f"--max_iters={iters} --lr_decay_iters={iters} "
            f"--eval_interval=250 --eval_iters=100 --log_interval=500")  # eval_iters=100 so val-loss comparisons between encodings are less noisy

        # %run leaves train.py's variables (model, optimizer, ...) in the notebook; free the GPU memory before the next run
        for name in ("model", "optimizer", "scaler", "raw_model", "X", "Y"):
            globals().pop(name, None)
        gc.collect()
        if DEVICE == "cuda": torch.cuda.empty_cache()


## 4. Classifier model (configurable pooling, head depth, partial freezing)


In [ ]:
import contextlib, torch, torch.nn as nn
from model import GPT, GPTConfig

# (use_config(), which switches the checkpoint / labels / results file for a (pos_enc, block size) pair, is defined in section 5.)

def build_gpt(pretrained=True):
    # Same architecture as the checkpoint; load the pre-trained weights, or leave them random for the scratch baseline.
    gpt = GPT(GPTConfig(**ckpt["model_args"]))
    if pretrained:
        sd = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model"].items()}   # strip torch.compile prefix if present
        gpt.load_state_dict(sd)
    return gpt

def make_head(n_embd, depth, n_classes=2, hidden=256, p=0.1):
    # depth 1: Dropout -> Linear(n_embd, n_classes)                       (linear probe)
    # depth d > 1: Dropout -> [Linear -> GELU -> Dropout] x (d-1) -> Linear(hidden, n_classes)
    # (first hidden layer is n_embd -> hidden, the others hidden -> hidden)
    assert depth >= 1
    layers, d = [nn.Dropout(p)], n_embd
    for _ in range(depth - 1):
        layers += [nn.Linear(d, hidden), nn.GELU(), nn.Dropout(p)]
        d = hidden
    layers.append(nn.Linear(d, n_classes))
    return nn.Sequential(*layers)

class AttnPool(nn.Module):
    # Learnable attention pooling: ONE learnable query vector scores every real token (scaled dot product),
    # softmax over the real tokens of each program, and the pooled vector is the weighted sum of the token states.
    def __init__(self, n_embd):
        super().__init__()
        self.query = nn.Parameter(torch.randn(n_embd) * 0.02)   # tiny init -> starts out close to mean pooling
        self.scale = n_embd ** -0.5
    def forward(self, x, mask):
        # x: (batch, length, n_embd), mask: (batch, length) 1 = real token, 0 = padding
        scores = (x @ self.query) * self.scale                          # (batch, length)
        scores = scores.masked_fill(mask == 0, float("-inf"))           # padding gets zero weight
        w = torch.softmax(scores, dim=-1)
        return (w.unsqueeze(-1) * x).sum(1)

class CodeClassifier(nn.Module):
    # GPT backbone + pooling + classification head.
    # n_freeze = number of bottom blocks to freeze (embeddings are frozen whenever n_freeze > 0;
    # n_freeze >= n_layer freezes the final LayerNorm too, i.e. the whole GPT).
    # pooling_mode: "mean" = mean over non-padding tokens | "last" = last non-padding token | "attn" = AttnPool (self.pool)
    def __init__(self, gpt, head_depth=1, hidden=256, n_classes=2, n_freeze=0, pooling_mode="mean"):
        super().__init__()
        assert pooling_mode in ("mean", "last", "attn"), pooling_mode
        self.gpt = gpt
        self.n_layer = len(gpt.transformer.h)
        self.n_freeze = min(n_freeze, self.n_layer)
        self.pooling_mode = pooling_mode
        self.pool = AttnPool(gpt.config.n_embd) if pooling_mode == "attn" else None
        self.head = make_head(gpt.config.n_embd, head_depth, n_classes, hidden)

        # Switch off gradients for everything that should stay frozen.
        # (The attention-pool query and the head are new, so they are always trainable.)
        t, k = gpt.transformer, self.n_freeze
        frozen = []
        if k > 0:
            frozen += [t.wte] + ([t.wpe] if "wpe" in t else []) + list(t.h[:k])   # embeddings + bottom k blocks
        if k >= self.n_layer:
            frozen.append(t.ln_f)                                                  # fully frozen: final LayerNorm too
        for mod in frozen:
            for p in mod.parameters():
                p.requires_grad = False

    def forward(self, idx, mask):
        # idx: (batch, length) token ids, mask: (batch, length) 1 for real tokens, 0 for padding (right padding)
        t, k = self.gpt.transformer, self.n_freeze
        no_grad = torch.no_grad() if k > 0 else contextlib.nullcontext()
        with no_grad:                                   # frozen bottom part: no backprop through it, so it is faster
            x = self.gpt.embed(idx)                     # wte (+ wpe for 'abs'); RoPE acts inside attention
            for block in t.h[:k]:
                x = block(x)
        for block in t.h[k:]:                           # trainable blocks
            x = block(x)
        if k >= self.n_layer:
            with torch.no_grad():
                x = t.ln_f(x)
        else:
            x = t.ln_f(x)
        # Pool over the real (non-padding) tokens -> one vector per program -> head -> class logits
        if self.pooling_mode == "mean":
            m = mask.unsqueeze(-1).float()
            pooled = (x * m).sum(1) / m.sum(1)
        elif self.pooling_mode == "last":
            last = mask.sum(1) - 1                      # right padding -> index of the last real token
            pooled = x[torch.arange(x.size(0), device=x.device), last]
        else:
            pooled = self.pool(x, mask)
        return self.head(pooled)


## Length-generalisation test

In [ ]:
# Length generalisation: LM loss on held-out code at context lengths beyond the training length (the block size).
# 'abs' cannot run past its block size (learned positions end there); 'rope' and 'nope' can -> this is where RoPE should differ.
import numpy as np, torch, torch.nn.functional as F
from model import GPT, GPTConfig

val_data = np.memmap("data/codeparrot/val.bin", dtype=np.uint16, mode="r")

def load_lm(pe, blk):
    ck = torch.load(f"{pretrain_dir(pe, blk)}/ckpt.pt", map_location="cpu", weights_only=False)
    m = GPT(GPTConfig(**ck["model_args"]))
    m.load_state_dict({k.replace("_orig_mod.", ""): v for k, v in ck["model"].items()})
    return m.to(DEVICE).eval()

@torch.no_grad()
def lm_loss_per_pos(m, T, n_batches=50, bs=16, seed=0):
    # bypasses GPT.forward (which asserts T <= block_size); returns the mean loss at every position
    g = torch.Generator().manual_seed(seed)
    out = []
    for _ in range(n_batches):
        ix = torch.randint(len(val_data) - T - 1, (bs,), generator=g)
        x = torch.stack([torch.from_numpy(val_data[i:i+T].astype(np.int64)) for i in ix]).to(DEVICE)
        y = torch.stack([torch.from_numpy(val_data[i+1:i+1+T].astype(np.int64)) for i in ix]).to(DEVICE)
        h = m.embed(x)
        for blk in m.transformer.h:
            h = blk(h)
        logits = m.lm_head(m.transformer.ln_f(h))
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1), reduction="none")
        out.append(loss.view(bs, T))
    return torch.cat(out).mean(0).cpu()

len_rows = []
for blk in BLOCK_SIZES:
    for pe in POS_ENCS:
        m = load_lm(pe, blk)
        for T in (256, 512, 1024):
            if "wpe" in m.transformer and T > m.config.block_size:
                print(f"b{blk} {pe:5s} T={T}: n/a (learned positions only up to {m.config.block_size})")
                continue
            pp = lm_loss_per_pos(m, T)
            last_q = pp[-T // 4:].mean().item()
            print(f"b{blk} {pe:5s} T={T}: mean loss {pp.mean():.3f} | last quarter of positions {last_q:.3f}")
            len_rows.append(dict(block_size=blk, pos_enc=pe, T=T, mean_loss=pp.mean().item(), last_quarter_loss=last_q))
        del m
        if DEVICE == "cuda": torch.cuda.empty_cache()
import pandas as pd
pd.DataFrame(len_rows).to_csv("length_generalisation.csv", index=False)


## 5. Data and training helpers

In [ ]:
def load_labels(blk):
    # (Re)build the train / test split from labels_xcodeeval_b{blk}.json. Sets the globals used by the helpers below.
    global rows_all, train_pool, test_rows, MAX_BALANCED
    rows_all = json.load(open(label_file(blk), encoding="utf-8"))

    # Split by problem (src_uid) so the same programming task never appears in both train and test.
    uids = sorted({r["group"] for r in rows_all})
    random.Random(0).shuffle(uids)
    test_uids = set(uids[:int(0.2 * len(uids))])           # 20% of problems -> test set
    test_rows  = [r for r in rows_all if r["group"] in test_uids]
    train_pool = [r for r in rows_all if r["group"] not in test_uids]
    random.Random(0).shuffle(train_pool)
    print(f"[block {blk}] pool: {len(train_pool)} train / {len(test_rows)} test")
    n_pos_tr = sum(r["label"] == 1 for r in train_pool); n_neg_tr = len(train_pool) - n_pos_tr
    MAX_BALANCED = 2 * min(n_pos_tr, n_neg_tr)
    print(f"train pool: {n_pos_tr} compile / {n_neg_tr} error -> largest class-balanced training set = {MAX_BALANCED}")
    n_pos_test = sum(r["label"] == 1 for r in test_rows)
    n_neg_test = len(test_rows) - n_pos_test
    print(f"test set: {n_pos_test} compile / {n_neg_test} error -> always guessing the majority class "
          f"scores {max(n_pos_test, n_neg_test)/len(test_rows):.3f} accuracy (balanced accuracy 0.500)")

def use_config(pe, blk):
    # Switch everything that depends on (positional encoding, block size): the checkpoint build_gpt() loads
    # (weights + model hyper-parameters written by train.py), the label file / train-test split, the max sequence length
    # used by make_batch, and the results file the grid writes to.
    global POS_ENC, BLOCK_SIZE, BLOCK, PRETRAIN_DIR, LABEL_FILE, RESULTS_CSV, ckpt
    POS_ENC, BLOCK_SIZE, BLOCK = pe, blk, blk
    PRETRAIN_DIR, LABEL_FILE, RESULTS_CSV = pretrain_dir(pe, blk), label_file(blk), results_csv(pe, blk)
    ckpt = torch.load(f"{PRETRAIN_DIR}/ckpt.pt", map_location="cpu", weights_only=False)
    load_labels(blk)

use_config(POS_ENC, BLOCK_SIZE)      # start with the first (encoding, block size); the grid loop switches through all of them

def get_train_rows(n_labeled):
    # Class-balanced subset: n_labeled/2 programs per class, or fewer if compile errors (the rare class) run out.
    # (So the effective size can be smaller than n_labeled.) None or "full" = whole pool, NOT balanced.
    # Always the same first-k examples per class, so a smaller set is contained in a larger one.
    if n_labeled is None or n_labeled == "full": return list(train_pool)
    pos = [r for r in train_pool if r["label"] == 1]
    neg = [r for r in train_pool if r["label"] == 0]
    k = min(n_labeled // 2, len(pos), len(neg))
    return pos[:k] + neg[:k]

def make_batch(rows):
    # Tokenize a list of examples, truncate to BLOCK tokens, right-pad to the longest in the batch.
    ids = [enc.encode_ordinary(r["code"])[:BLOCK] for r in rows]
    L = max(len(i) for i in ids)
    x = torch.zeros(len(ids), L, dtype=torch.long)
    m = torch.zeros(len(ids), L, dtype=torch.long)
    for k, i in enumerate(ids):
        x[k, :len(i)] = torch.tensor(i); m[k, :len(i)] = 1     # right-padded; causal attention => real tokens never see padding
    y = torch.tensor([r["label"] for r in rows])
    return x.to(DEVICE), m.to(DEVICE), y.to(DEVICE)

def metrics_from_confusion(conf):
    # conf[true, pred] 2x2 counts; class 0 = compilation error, class 1 = compiles.
    # Precision of a class nobody was predicted as is defined as 0 (sklearn's default), so f1 is 0 there too.
    conf = conf.double()
    out = {"test_acc": conf.diag().sum().item() / conf.sum().item()}
    recs, f1s = [], []
    for c, name in ((0, "error"), (1, "compile")):
        tp, n_true, n_pred = conf[c, c].item(), conf[c].sum().item(), conf[:, c].sum().item()
        rec  = tp / n_true if n_true else float("nan")
        prec = tp / n_pred if n_pred else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        out[f"recall_{name}"], out[f"precision_{name}"], out[f"f1_{name}"] = rec, prec, f1
        recs.append(rec); f1s.append(f1)
    valid = [r for r in recs if r == r]                         # drop nan (class absent from the test set)
    out["bal_acc"]  = sum(valid) / len(valid)                   # balanced accuracy = mean of the per-class recalls
    out["macro_f1"] = sum(f1s) / len(f1s)
    return out

def evaluate(model, rows):
    # Returns a dict: test_acc, bal_acc, recall_/precision_/f1_ for error and compile, macro_f1.
    # Balanced accuracy = mean of the per-class recalls, so a classifier that always predicts the
    # majority class scores 0.5 even when the test set is imbalanced.
    model.eval(); conf = torch.zeros(2, 2, dtype=torch.long)
    eval_bs = max(1, 64 * 256 // BLOCK)      # ~16k tokens per eval step at any block size
    with torch.no_grad():
        for i in range(0, len(rows), eval_bs):
            x, m, y = make_batch(rows[i:i+eval_bs])
            pred = model(x, m).argmax(-1)
            conf += torch.bincount(y * 2 + pred, minlength=4).view(2, 2).cpu()
    model.train()
    return metrics_from_confusion(conf)

def train_classifier(pretrained=True, head_depth=None, n_labeled=None, epochs=None,
                     n_freeze=None, pooling=None, bs=16, seed=0, verbose=True):
    # Train one classifier and return (model, history). history = one dict per epoch:
    #   epoch (1-based), train_loss (mean of the training-batch losses during that epoch), and the test metrics
    #   from evaluate(): test_acc, bal_acc, recall_error, recall_compile, precision_error, precision_compile,
    #   f1_error, f1_compile, macro_f1.
    # Any argument left as None falls back to the config cell. (bs = fine-tuning batch size.)
    # pretrained=False -> random-init backbone (baseline). seed changes head init, dropout and data order
    # (the labelled subset itself is fixed).
    head_depth = HEAD_DEPTH if head_depth is None else head_depth
    n_labeled  = N_LABELED if n_labeled is None else n_labeled
    epochs     = FT_EPOCHS if epochs is None else epochs
    n_freeze   = N_FREEZE if n_freeze is None else n_freeze
    pooling    = POOLING if pooling is None else pooling

    torch.manual_seed(seed); random.seed(seed)
    rows = get_train_rows(n_labeled)
    bs = min(bs, max(1, len(rows)))
    model = CodeClassifier(build_gpt(pretrained), head_depth, HEAD_HIDDEN, n_freeze=n_freeze, pooling_mode=pooling).to(DEVICE)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"pretrained={pretrained} pooling={pooling} depth={head_depth} freeze={n_freeze}/{model.n_layer} "
          f"train={len(rows)} seed={seed} trainable={n_tr/1e6:.2f}M")

    # Two learning rates: the new head (and the attention-pool query, if any) learns fast, the (unfrozen) GPT blocks change slowly.
    head_params = list(model.head.parameters())
    if pooling == "attn": head_params += list(model.pool.parameters())
    gpt_params = [p for p in model.gpt.parameters() if p.requires_grad]   # empty when fully frozen
    groups_ = [{"params": head_params, "lr": LR_HEAD}]
    if gpt_params:
        groups_.append({"params": gpt_params, "lr": LR_BACKBONE if pretrained else LR_BACKBONE_SCRATCH})
    opt = torch.optim.AdamW(groups_, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss()

    history = []
    model.train()
    for epoch in range(epochs):
        random.shuffle(rows)
        loss_sum, n_seen = 0.0, 0
        for step, i in enumerate(range(0, len(rows), bs)):
            x, m, y = make_batch(rows[i:i+bs])
            loss = loss_fn(model(x, m), y)
            opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item() * len(y); n_seen += len(y)
            if verbose and step % 50 == 0:
                print(f"  epoch {epoch+1} step {step} loss {loss.item():.3f}")
        history.append(dict(epoch=epoch + 1, train_loss=loss_sum / n_seen, **evaluate(model, test_rows)))
        if verbose:
            h = history[-1]
            print(f"epoch {h['epoch']}: train loss {h['train_loss']:.3f} test acc {h['test_acc']:.3f} "
                  f"(balanced {h['bal_acc']:.3f}, macro-F1 {h['macro_f1']:.3f})")
    return model, history


## 6. Single runs

In [ ]:
# Two runs using the settings in the config cell, on the currently active (positional encoding, block size)
# (the first of each until the grid loop has switched; call use_config(pe, blk) to pick another):
model_pre, hist_pre = train_classifier(pretrained=True)
model_scratch, hist_scratch = train_classifier(pretrained=False, n_freeze=0)
import pandas as pd
print(pd.DataFrame(hist_pre).round(3).to_string(index=False))


## 7. Experiment grid: block size x positional encoding x pooling x initialisation x frozen blocks x head depth x labelled-data size x seed


In [ ]:
import pandas as pd, time

# CSV schema: one row per (run, epoch). init = "pretrained" / "scratch"; n_labeled = the requested size as a string
# (10, 100, 1000, 10000, full); n_train = number of training examples actually used; run_min = wall time of the whole run.
CSV_COLS = ["pos_enc", "block_size", "pooling", "depth", "n_freeze", "n_labeled", "seed", "epoch",
            "train_loss", "test_acc", "bal_acc", "recall_error", "recall_compile",
            "precision_error", "precision_compile", "f1_error", "f1_compile", "macro_f1",
            "init", "n_train", "run_min"]
KEY_COLS = ["init", "pos_enc", "block_size", "pooling", "depth", "n_freeze", "n_labeled", "seed"]   # identifies one run

def run_grid():
    # Full grid for the CURRENT (positional encoding, block size) (see the "What is being compared" table).
    # Seeds are the outer loop, so if you stop early you still have complete grids for the first seeds.
    # After every finished run its per-epoch rows are APPENDED to RESULTS_CSV; runs already in the file
    # (all GRID_EPOCHS epochs present) are skipped, so an interrupted grid can simply be restarted.
    for n in GRID_SIZES:
        if isinstance(n, int) and n > MAX_BALANCED:
            print(f"note: n_labeled={n} is larger than the largest balanced set -> uses {MAX_BALANCED} examples (see n_train)")
    jobs = []
    for seed in GRID_SEEDS:
        for pooling in POOLINGS:
            for k in GRID_FREEZE:
                for depth in GRID_DEPTHS:
                    for n in GRID_SIZES:
                        jobs.append(dict(init="pretrained", pretrained=True, n_freeze=k, pooling=pooling,
                                         depth=depth, n_labeled=n, seed=seed))
            for depth in GRID_DEPTHS:                      # random-init baseline, fully trainable
                for n in GRID_SIZES:
                    jobs.append(dict(init="scratch", pretrained=False, n_freeze=0, pooling=pooling,
                                     depth=depth, n_labeled=n, seed=seed))
    key = lambda j: (j["init"], POS_ENC, BLOCK_SIZE, j["pooling"], j["depth"], j["n_freeze"], str(j["n_labeled"]), j["seed"])

    done = set()
    if os.path.exists(RESULTS_CSV):
        prev = pd.read_csv(RESULTS_CSV, dtype={"n_labeled": str})
        epochs_done = prev.groupby(KEY_COLS)["epoch"].max()
        done = {tuple(k) for k, e in epochs_done.items() if e >= GRID_EPOCHS}
    todo = [j for j in jobs if key(j) not in done]
    print(f"{len(jobs)} runs in this grid, {len(jobs) - len(todo)} already in {RESULTS_CSV}, {len(todo)} to do")

    t0 = time.time()
    for j, job in enumerate(todo, 1):
        t_run = time.time()
        _, hist = train_classifier(pretrained=job["pretrained"], n_freeze=job["n_freeze"], head_depth=job["depth"],
                                   n_labeled=job["n_labeled"], epochs=GRID_EPOCHS, pooling=job["pooling"],
                                   seed=job["seed"], verbose=False)
        run_min = (time.time() - t_run) / 60
        n_train = len(get_train_rows(job["n_labeled"]))
        rows = [dict(pos_enc=POS_ENC, block_size=BLOCK_SIZE, pooling=job["pooling"], depth=job["depth"],
                     n_freeze=job["n_freeze"], n_labeled=str(job["n_labeled"]), seed=job["seed"],
                     init=job["init"], n_train=n_train, run_min=run_min, **h) for h in hist]
        pd.DataFrame(rows, columns=CSV_COLS).to_csv(RESULTS_CSV, mode="a", index=False,
                                                    header=not os.path.exists(RESULTS_CSV))   # save after EVERY run (safe if interrupted)
        h = hist[-1]
        print(f"[{j}/{len(todo)}] {job['init']} pool={job['pooling']} freeze={job['n_freeze']} depth={job['depth']} "
              f"n={job['n_labeled']} seed={job['seed']}: acc={h['test_acc']:.3f} bal_acc={h['bal_acc']:.3f} "
              f"rec_err={h['recall_error']:.3f} rec_ok={h['recall_compile']:.3f} macro_f1={h['macro_f1']:.3f}  "
              f"elapsed {(time.time()-t0)/60:.1f} min")

for blk in BLOCK_SIZES:             # outer loop: block size (own checkpoints + labels); inner loop: positional encoding
    for pe in POS_ENCS:             # one full grid per (block size, encoding); each writes grid_results_<pe>_b<block>.csv
        use_config(pe, blk)
        print(f"\n===== fine-tuning grid: block_size={blk} pos_enc={pe} =====")
        run_grid()


In [ ]:
# Summary tables: rows = (block size, positional encoding, pooling, init, frozen blocks, head depth), columns = labelled examples.
# Reads every grid_results_<pe>_b<block>.csv present and uses the LAST epoch of each run.
# Balanced accuracy is the one to trust if the test set is imbalanced (see the "test set:" line above).
# Differences of ~1-2 points are within noise with this test-set size; check the std across seeds.
# (The CSVs hold every epoch, so learning curves are one groupby away, e.g. df.groupby(["pooling","epoch"])["bal_acc"].mean().)
import glob
pd.set_option("display.max_rows", None, "display.width", 220)
df = pd.concat([pd.read_csv(f, dtype={"n_labeled": str}) for f in sorted(glob.glob("grid_results_*_b*.csv"))], ignore_index=True)
final = df[df["epoch"] == df.groupby(KEY_COLS)["epoch"].transform("max")]
size_cols = [str(s) for s in GRID_SIZES]
IDX = ["block_size", "pos_enc", "pooling", "init", "n_freeze", "depth"]
tab = lambda col, f: final.pivot_table(index=IDX, columns="n_labeled", values=col, aggfunc=f).reindex(columns=size_cols)
print("mean BALANCED accuracy over seeds"); print(tab("bal_acc", "mean").round(3))
print("\nstd of balanced accuracy over seeds"); print(tab("bal_acc", "std").round(3))
print("\nmean plain accuracy over seeds"); print(tab("test_acc", "mean").round(3))
print("\nmean macro-F1 over seeds"); print(tab("macro_f1", "mean").round(3))

print("\nmean recall of the ERROR class (label 0)"); print(tab("recall_error", "mean").round(3))
print("\nmean recall of the COMPILE class (label 1)"); print(tab("recall_compile", "mean").round(3))
print("\nmean training time per run [min]"); print(tab("run_min", "mean").round(2))
